# 2.1 - Quality Control - Delaunay Triangulation and Neighbor Mapping

Este notebook constrói a malha de circunvizinhos utilizando **Triangulação de Delaunay** com exclusão da estação-alvo (**Point-Out**).

**Conceito Point-Out:**
- A estação alvo é removida do conjunto de coordenadas;
- Constrói-se a triangulação de Delaunay sobre as demais estações;
- Localiza-se o triângulo (simplex) que envolve geometricamente a coordenada do alvo;
- Os 3 vértices deste triângulo constituem os circunvizinhos mais representativos.

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.spatial import Delaunay
from shapely.geometry import Point, Polygon

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
sys.path.insert(0, str(PROJECT_ROOT.parent))

from src.config import ROOT
from src.utils.logger import save_result

RESULTS_DIR = ROOT / 'src' / '5 - Results'

## 1. Carregamento dos Metadados Espaciais das Estações

In [ ]:
props_path = RESULTS_DIR / 'annual_gauge_properties.parquet'
df_props = pd.read_parquet(props_path)
coords_df = df_props[['gauge_code', 'latitude', 'longitude']].drop_duplicates().dropna()
coords_df = coords_df.reset_index(drop=True)
print(f'Total de estações com coordenadas válidas: {len(coords_df):,}')
coords_df.head()

## 2. Construção da Malha Delaunay Point-Out

In [ ]:
neighbors_records = []
n_total = len(coords_df)

for idx, row in coords_df.iterrows():
    target_code = row['gauge_code']
    target_pt = Point(row['longitude'], row['latitude'])
    
    # Base reduzida excluindo a própria estação alvo
    ref_df = coords_df[coords_df['gauge_code'] != target_code].reset_index(drop=True)
    if len(ref_df) < 3:
        continue
    
    pts = ref_df[['longitude', 'latitude']].values
    try:
        tri = Delaunay(pts)
        s_idx = tri.find_simplex(np.array([[target_pt.x, target_pt.y]]))[0]
        if s_idx >= 0:
            vertex_indices = tri.simplices[s_idx]
            neighbor_codes = ref_df.iloc[vertex_indices]['gauge_code'].tolist()
            neighbors_records.append({
                'gauge_code': target_code,
                'neighbor_1': neighbor_codes[0],
                'neighbor_2': neighbor_codes[1],
                'neighbor_3': neighbor_codes[2],
                'is_boundary': False
            })
        else:
            neighbors_records.append({
                'gauge_code': target_code,
                'neighbor_1': None,
                'neighbor_2': None,
                'neighbor_3': None,
                'is_boundary': True
            })
    except Exception:
        pass

df_neighbors = pd.DataFrame(neighbors_records)
print(f'Mapeamento concluído: {len(df_neighbors):,} estações')
print('Estações de borda:', df_neighbors['is_boundary'].value_counts().to_dict())
df_neighbors.head()

## 3. Persistência dos Circunvizinhos em Parquet

In [ ]:
out_path = save_result(df_neighbors, 'gauge_delaunay_neighbors.parquet', subdir='5 - Results')
print('Salvo em:', out_path)